***

# Question 1: Clinic Queue Simulation

## Supplementary Jupyter Notebook for Mathematical Modelling for Sustainable Development Coursework

***

Methods and assumptions are explained/modelled throughout. Inline comments are left to show how I coded this logically and not meant to be explanative - the markdown blocks serve this purpose!


Full development of this code can be found in the following link: 
[Link to Github Repo](https://github.com/Leonie-G-B/MathModSusDev)


> Student Num. 2101377

> Email: ch21886@bristol.ac.uk

***

### Nomenclature and utility functions: 

Getting this out the way to reduce code clutter later on

In [ ]:
##### Standard inputs & type hinting helpers
import numpy as np
import pandas as pd
from typing import Literal, TypedDict, Sequence
from enum import StrEnum

# Simulation specifics
from collections import deque

# Plots
import seaborn as sbn
import matplotlib.pyplot as plt


##### Type hinting classes
class ServiceMethods(StrEnum): 
    EXPONENTIAL = "exponential"
    LOGNORMAL = "lognormal"    
    NORMAL = "normal"

class ClinicianConfig(TypedDict): # define a type of shift for n number of clinicians
    shift_pattern : tuple[float, float] # list of (start, end)
    appointment_length : float # in minutes, list must match 


##### Utility functions 

def format_time_hours(val: float) -> str:
    "Convert decimal hr numbers into something human readable and clear."
    if val >= 1.0:
        hours = int(val)
        minutes = int(round((val - hours) * 60))
        return f"{hours}h {minutes}m" if minutes > 0 else f"{hours}h"
    else:
        minutes = int(round(val * 60))
        return f"{minutes}mins"

***

## Key objects: Clinician and Simulation

### Summary of assumptions and simulation architecture 
* Service times & arrival rates (mu, lambda) are randomly and memoryless. See more info & plots on this on subsequent blocks.
* Single queue, First Come First Served (FCFS). The size of the 'waiting room' has not been restricted and assumed infinite. 
* The simulation is a fixed duration "day" of the clinic. The time of the start & end of the day (plus the clinicians' work schedules) are required inputs. This was done to allow the investigation into changing the schedules of the clinicians throughout a day of work clearly.


Note: This block is lengthy as it is the complete Clinic Simulation for all subsequent blocks.

***

In [ ]:

class Clinician: 
    def __init__(self, id, appointment_time: int, 
                 shift_start: float,
                 shift_end: float):
        self.id : int = id
        self.mu : float = 60/appointment_time
        
        self.shift_start : float = shift_start
        self.shift_end   : float = shift_end

        self.available : bool = True
        self.next_available: float = None #float time of when they are next free

        self.total_appointment_time : float = 0.0
        self.total_downtime : float = 0.0
        self.last_event_time : float = shift_start #to calculate downtime between appointments
    

    def appointment_start(self, current_time: float): 
        self.total_downtime += current_time - self.last_event_time #assuming last event is finihsing an appointment

        self.available = False
        self.last_event_time = current_time

    def appointment_end(self, current_time: float):
        self.total_appointment_time += current_time - self.last_event_time #we could assume apppointment length but this is more foolproof incase of different EOD behaviour

        self.available = True
        self.last_event_time = current_time
    
    def calc_workload(self):
        shift_length = self.shift_end - self.shift_start
        workload = (shift_length - self.total_downtime) / shift_length
        self.workload = workload


class ClinicSim: 
    def __init__(sim, lambda_base: int, appointment_time: int, 
                 service_method: ServiceMethods = "exponential",
                 peak_multiplier: int = None, #should be 2,4,8 - use checking?
                 open_close: tuple[float, float] = (8.0, 17.5),
                 peak_hrs: tuple[float, float] = (10.0, 14.0), 
                 peak_normal_shape: bool = False, #if True, then creates a normal curve within the peak with max value peak_multiplier. Otherwise uses other default methods.
                 **kwargs):
        
        sim.lambda_base = lambda_base
        sim.lambdas_t = []
        
        sim.peak_multiplier: int = peak_multiplier
        sim.normal_peak: bool = peak_normal_shape
        sim.peak_start = peak_hrs[0]
        sim.peak_end = peak_hrs[1]

        sim.mu = 60/appointment_time #hourly rate

        sim.open_time = open_close[0]
        sim.close_time = open_close[1]

        sim.clock = sim.open_time # start at the start!
        
        sim.queue = deque()
        sim.waits = []# list of the waiting times

        sim.num_in_system = 0
        sim.sys_state = [(sim.clock,0)] #tuples, (time, no. patients in queue) 

        sim.arrival_times = []
        sim.departure_times = [] #just for data logging reasons

        sim.self_set_service_method(service_method, **kwargs) #can pass in "logn_simga" for example for lognormal serv. method

        # sim.servers = [None] * sim.num_clinicians #none indicates free server
        sim.clinicians: list[Clinician] = []

        sim.t_arrival = sim.clock + sim.generate_interarrival()


    def self_set_service_method(sim, method: ServiceMethods, **kwargs): 
        mean_serv_hrs = 1 /sim.mu
        if method == "exponential": 
            sim._service_func = lambda: np.random.exponential(mean_serv_hrs)
        elif method == "lognormal": 
            sigma = kwargs.get("logn_sigma", 0.5)
            mu_log = np.log(mean_serv_hrs) - 0.5 * sigma**2
            sim._service_func = lambda: np.random.lognormal(mean= mu_log, sigma = sigma)
        elif method == "normal": 
            sim._service_func = lambda: max(0, np.random.normal(
                loc = mean_serv_hrs,
                scale= kwargs.get("norm_scale", 0.2) * mean_serv_hrs
                ))

    ####################################################################

    def create_clinicians(sim, n_clinicians: int, config: ClinicianConfig):
        cur_in_list = len(sim.clinicians)
        for i in range(n_clinicians):
            sim.clinicians.append(
                Clinician(
                    id = i + cur_in_list,
                    appointment_time=config["appointment_length"],
                    shift_start=config["shift_pattern"][0],
                    shift_end=config["shift_pattern"][1]
                )
            )

    ####################################################################

    def get_lambda(sim):
        if sim.peak_start <= sim.clock <= sim.peak_end: #if during peak time
            if sim.peak_multiplier is not None: 
                multiplier = sim.peak_multiplier
                if sim.normal_peak: 
                    t = sim.clock 
                    centre = (sim.peak_start + sim.peak_end) /2
                    half_width = (sim.peak_end - sim.peak_start)/ 2
                    x = (t - centre) / half_width
                    k = 3 #steepness - should this be an input? 

                    shape = np.exp(-k * x ** 2)
                    edge = np.exp(-k) #normalise
                    shape = (shape - edge) / (1- edge)
                    multiplier = 1 + (sim.peak_multiplier - 1) * shape
            else:
                multiplier = np.random.choice([2,3,4]) #" the number of patient arrivals can douple, triple, or even quadruple"
            return sim.lambda_base * multiplier
        return sim.lambda_base

    def generate_interarrival(sim):
        lam = sim.get_lambda()
        sim.lambdas_t. append((lam, sim.clock))
        return np.random.exponential(1 / lam)

    def generate_service(sim):
        return sim._service_func()
    
    def get_free_clinician(sim): 
        for c in sim.clinicians: 
            if c.available and sim.clock >= c.shift_start and sim.clock <= c.shift_end:
                return c
        return None #i.e. no one is free!
    
    ####################################################################

    def arrival(sim):
        sim.num_in_system += 1
        sim.queue.append(sim.clock)
        sim.arrival_times.append(sim.clock)

        clinician = sim.get_free_clinician()

        if clinician is not None:
            arrival_time = sim.queue.popleft()

            clinician.appointment_start(sim.clock)

            service_time = sim.generate_service()
            clinician.next_available = sim.clock + service_time

            wait = sim.clock - arrival_time
            sim.waits.append(wait)

        sim.t_arrival = sim.clock + sim.generate_interarrival()

    def departure(sim, clinician: Clinician): #any mutation to clinician here will modify the original sim.clinician object 
        sim.num_in_system -= 1
        sim.departure_times.append(sim.clock)

        clinician.appointment_end(sim.clock)

        if sim.queue:
            arrival_time = sim.queue.popleft()

            clinician.appointment_start(sim.clock)

            service_time = sim.generate_service()
            clinician.next_available = sim.clock + service_time

            wait = sim.clock - arrival_time
            sim.waits.append(wait)
        else:
            clinician.next_available = None

        if sim.clock > clinician.shift_end: #enforce end of shift?
            clinician.next_available = None
            clinician.available = False

    def get_next_departure(sim): #helper function
        active = [
            (c.next_available, c)
            for c in sim.clinicians if c.next_available is not None
        ]
        return min(active, default=(float('inf'), None), key=lambda x:x[0])
        #return next availabe and clinician object (find smallest first element in list and replace with a default value of 'inf' if none)


    ####################################################################

    def step(sim): #discrete time event - we just jump to the next time where *something* happens
        assert len(sim.clinicians) >= 1, "No clinicians created. Call create_clinicians()." #kept making this mistake !!
        next_depart_time, clinician = sim.get_next_departure()

        if sim.t_arrival <= next_depart_time and sim.t_arrival <= sim.close_time: #if arrival happens next (before available server) and its before closing
            sim.clock = sim.t_arrival
            sim.arrival() #jump to arrival time and initiate arrival 
        else:
            sim.clock = next_depart_time
            if clinician is not None: 
                sim.departure(clinician)

        sim.sys_state.append((sim.clock, sim.num_in_system)) # Record system state (queue length OR total system)


*** 

## Run a simple simulation 

So that the following blocks of code make sense.
This simulation is a simple M/M/6 queue simulation. Markovian arrivals and departures (service), with 6 clinicians available for the full day, aiming for 30min appointment times, FCFS. The base arrival rate /hour is 8, ignoring effect of 'peak hours'. This simulation represents a single day of the clinic (i.e. no multisim averaging - this is not going to be used to quote metrics at this point).

In [ ]:
simple_sim_params = {
    "lambda_base" : 8,
    "appointment_time" : 30,
    "service_method" : "exponential",
    "peak_multiplier" : 1
} 

simple_sim = ClinicSim(**simple_sim_params)

simple_sim.create_clinicians( #instantiate the aforementioned clinician config
    n_clinicians= 6, 
    config= {
        "shift_pattern" : (8.0, 17.5),
        "appointment_length" : 30
    }
)

while simple_sim.clock < simple_sim.close_time: 
    simple_sim.step()


***

## Service and Arrival Rate

I modelled a few different options here and observed the resultant trends to balance what maintains the original assumptions, and what is realistic. 

We are going to plot what the serice rate (distribution of appointment times) would look like for that sim.This is using the same method as the ClinicSim instead of retrieving the small sample size of appointments from the actual sim, or separately recreating the distribution - this is helpful from a coding and data visualisation angle.
The arrival rate is constant for this sim.


In [ ]:
def plot_service_distribution_actual(sim: ClinicSim, n_samples: int = 500):

    samples = [sim.generate_service() * 60 for _ in range(n_samples)]

    fig, ax = plt.subplots(figsize=(12, 6))

    #gonna convert things from mu to times (in minutes)
    ax.hist(samples, bins=40, density=True, alpha=0.6, color="steelblue",
            edgecolor="black", label="Sampled service times")

    try: 
        sbn.kdeplot(samples, ax=ax, color="darkred", linewidth=2,
                    label="KDE (smooth density)")
    except Exception: 
        print("failed to plot seaborn kde bounds")

    ax.set_xlabel("Service time")
    ax.set_ylabel("Density")
    ax.set_title(f"Service Time Distribution. N_samples = {n_samples}")

    mean_val = np.mean(samples)
    ax.axvline(mean_val, color="green", linestyle="--", linewidth=2,
               label=f"Mean = {mean_val:.2f}")

    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_xlim(left = 0)
    ax.legend()

    fig.tight_layout()
    # print("completed plot") #For a debugging breakpoint

plot_service_distribution_actual(sim = simple_sim, n_samples = 1000)

***

## Alternative Appointment distribution modelling

I opted to instead use a different distribution for my model, as the above exponential distribution had too many very very short appointments to what I felt was realistic. I modelled a lognormal distribution and was pleased with this instead, making this, and subsequent, simulations a M/G/c queue. 

## Peak hours behaviour 

In order to emulate the behaviour of the 'peak hours' between 10.00 and 14.00hrs, I considered a random multiplier effect (coupled with a random base lambda value) but instead wanted more clarity in the affect of arrival rates on the clinic. Thus, I created an arrival rate distibution that follows a general normal bell curve within the peak window, acting as a multiplier to the base lambda. 

Note: The arrival rate plot *is* the actual service rates from the sim, so has the sampling resolution of the simulation itself, hence why not a smooth curve, but sufficiently conveys what I need it to here!

### Both choices are demonstrated in the plots below

In [ ]:

def plot_lamda_t(sim: ClinicSim): #This DOES plot the actual lambdas from the simulation!!

    lambdas, times = zip(*sim.lambdas_t)
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.step(times, lambdas, where='post', linewidth=2, label="λ(t) arrival rate")

    mean_lambda = np.mean(lambdas)
    ax.axhline(mean_lambda, color='red', linestyle='--', linewidth=1.5,
               label=f"Mean λ = {mean_lambda:.2f}")

    if hasattr(sim, "peak_start") and hasattr(sim, "peak_end"): #if provided, plot the peak hrs
        ax.axvspan(sim.peak_start, sim.peak_end, color='yellow', alpha=0.2,
                   label="Peak hours")

    ax.set_xlabel("Time of Day")
    ax.set_ylabel("Arrival Rate λ(t)")
    ax.set_title("Arrival Rate Over Time")

    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc="upper left")
    fig.tight_layout()

    print("completed plot")

#########################################################

simplev2_sim_params = {
    "lambda_base" : 8,
    "appointment_time" : 30,
    "service_method" : "lognormal",
    "peak_multiplier" : 4, # Up to 4x the baseline arrival rate at the peak time
    "peak_normal_shape": True
} 


simplev2_sim = ClinicSim(**simplev2_sim_params)

simplev2_sim.create_clinicians( #instantiate the aforementioned clinician config
    n_clinicians= 6, 
    config= {
        "shift_pattern" : (8.0, 17.5),
        "appointment_length" : 30
    }
)

while simplev2_sim.clock < simplev2_sim.close_time: 
    simplev2_sim.step()


plot_service_distribution_actual(sim = simplev2_sim, n_samples = 1000)
plot_lamda_t(sim= simplev2_sim)

***

## Finally, a sanity check plot

To show the arrivals, departures, and total patients in system throughout one simulation.

This plot clearly shows some interesting things about this system - it really struggles to recoever the backlog from the peak hours, so this is where the suggested improvements are centred around!.

In [ ]:


def plot_arrival_depart(sim: ClinicSim):

    _, ax = plt.subplots(figsize=(12,6))

    times, values = zip(*sim.sys_state)

    ax.step(times, values, where='post', label="Patients in system")

    arrival_sorted = np.sort(sim.arrival_times)
    departure_sorted = np.sort(sim.departure_times)

    ax.step(arrival_sorted, np.arange(1, len(arrival_sorted)+1),
            where='post', linestyle='--', label="Cumulative arrivals")
    ax.step(departure_sorted, np.arange(1, len(departure_sorted)+1),
            where='post', linestyle=':', label="Cumulative departures")

    ax.axvspan(sim.peak_start, sim.peak_end, alpha=0.2) # peak hours

    ax.set_xticks(np.arange(sim.open_time, sim.close_time + 0.5, 0.5)) #grid (30min intervals)
    ax.grid(True, which='both', axis='x', linestyle='--', alpha=0.5)

    ax.legend()
    ax.set_title(f"Patient arrival, departures, and total system capacity")

    # print("Finished plotting")


plot_arrival_depart(sim = simplev2_sim)